In [ ]:
from google.colab import drive
from pathlib import Path
import unicodedata
import pandas as pd

drive.mount("/content/drive")

In [ ]:
folder = Path(
    "/content/drive/MyDrive/KB_FinalProject_Team6"
)

filename = "인허가_파생지표_전체_202301_202604.parquet"

matches = [
    path
    for path in folder.rglob("*.parquet")
    if unicodedata.normalize("NFC", path.name) == filename
]

if len(matches) != 1:
    raise ValueError(
        f"파일이 {len(matches)}개 발견됐습니다. 경로를 확인하세요.\n"
        + "\n".join(map(str, matches))
    )

parquet_path = matches[0]
print("사용 파일:", parquet_path)

In [ ]:
df = pd.read_parquet(parquet_path)

print("행, 열:", df.shape)
display(df.head(10))

print("\n전체 컬럼")
print(df.columns.tolist())

print("\n월별 행 수")
display(df.groupby("base_ym").size().rename("점포수"))

print("\n메모리 사용량")
print(f"{df.memory_usage(deep=True).sum() / 1024**3:.2f} GB")

In [ ]:
features = [
    "age_m", "age_band", "is_new",
    "biz_type", "is_franchise",
    "dong_age_med", "closed_life_avg",
    "surv_6m", "fr_share",
    "close_rate_6m", "open_rate_6m", "net_6m",
    "biz_entropy", "same_300m", "close_100m_6m",
]

display(df[["base_ym", "점포이력ID"] + features].head(20))

display(
    df[features].isna().mean()
    .rename("결측률")
    .to_frame()
)

In [ ]:
column_map = {
    # 점포 기본 정보
    "base_ym": "기준월",
    "open_s": "개업일",
    "close_s": "폐업일",
    "lon": "경도",
    "lat": "위도",
    "coord_valid": "좌표유효여부",

    # 파생지표
    "age_m": "업력개월",
    "age_band": "업력구간",
    "is_new": "신규점포여부",
    "biz_type": "세부업종",
    "is_franchise": "프랜차이즈여부",
    "dong_age_med": "행정동업력중앙값",
    "closed_life_avg": "폐업점포평균영업개월",
    "surv_6m": "개업코호트육개월생존율",
    "fr_share": "동종프랜차이즈비중",
    "close_rate_6m": "최근육개월폐업률",
    "open_rate_6m": "최근육개월개업률",
    "net_6m": "최근육개월점포순증",
    "biz_entropy": "행정동업종다양성",
    "same_300m": "반경삼백미터동종영업점수",
    "close_100m_6m": "반경백미터최근육개월폐업점수",

    # 정답 및 표본 수
    "label_close_6m": "향후육개월폐업여부",
    "n_dong_age": "행정동업력집계점포수",
    "n_hk": "행정동동종영업점수",
    "n_gk": "구군동종영업점수",
    "n_entropy": "업종다양성집계점포수",
}

# 행정동·구군별 보정 전 지표와 관측 수
for level, region in [("h", "행정동"), ("g", "구군")]:
    column_map.update({
        f"fr_share_{level}_raw":
            f"{region}동종프랜차이즈비중보정전",
        f"close_rate_6m_{level}_raw":
            f"{region}최근육개월폐업률보정전",
        f"open_rate_6m_{level}_raw":
            f"{region}최근육개월개업률보정전",
        f"net_6m_{level}_raw":
            f"{region}최근육개월점포순증보정전",
        f"closed_life_avg_{level}_raw":
            f"{region}폐업점포평균영업개월보정전",
        f"surv_6m_{level}_raw":
            f"{region}개업코호트육개월생존율보정전",
        f"n_fr_{level}":
            f"{region}동종프랜차이즈여부관측점포수",
        f"n_start_{level}k":
            f"{region}동종육개월전영업점수",
        f"n_open_{level}k":
            f"{region}동종최근육개월개업점수",
        f"n_close_{level}k":
            f"{region}동종최근육개월폐업점수",
        f"n_closed_life_{level}":
            f"{region}동종최근일년폐업점수",
        f"n_cohort_{level}":
            f"{region}동종생존율집계개업점수",
    })

# 점포이력ID → 점포이력식별자 등 기존 영문 약어도 변경
column_map.update({
    "점포이력ID": "점포이력식별자",
    "통합점포ID": "통합점포식별자",
    "점포ID": "점포식별자",
})

new_columns = [column_map.get(col, col) for col in df.columns]

# 매핑되지 않은 영문 컬럼 및 중복 검사
import re

unmapped = [
    col for col in new_columns
    if re.search(r"[A-Za-z]", col)
]

if unmapped:
    raise ValueError(f"한글 이름을 추가로 지정해야 하는 컬럼: {unmapped}")

if len(new_columns) != len(set(new_columns)):
    raise ValueError("변경 후 컬럼명이 중복됩니다.")

df.columns = new_columns

print("한글 컬럼명 변경 완료")
print(df.columns.tolist())
display(df.head(10))

In [ ]:
df.tail()

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
easy_names = {
    "점포이력식별자": "점포번호",
    "통합점포식별자": "통합점포번호",
    "점포식별자": "인허가번호",

    "업력개월": "영업기간_개월",
    "업력구간": "영업기간_구간",
    "신규점포여부": "개업1년미만",
    "세부업종": "점포업종",
    "프랜차이즈여부": "프랜차이즈",

    "행정동업력중앙값": "동네점포_영업기간중앙값_개월",
    "폐업점포평균영업개월": "주변동종_폐업전평균영업기간_개월",
    "개업코호트육개월생존율": "신규점포_6개월생존율",
    "동종프랜차이즈비중": "주변동종_프랜차이즈비율",
    "최근육개월폐업률": "주변동종_6개월폐업률",
    "최근육개월개업률": "주변동종_6개월개업률",
    "최근육개월점포순증": "주변동종_6개월점포증감",
    "행정동업종다양성": "동네_업종다양성",

    "반경삼백미터동종영업점수": "300미터내_동종점포수",
    "반경백미터최근육개월폐업점수": "100미터내_6개월폐업수",

    "향후육개월폐업여부": "앞으로6개월내_폐업여부",
    "행정동업력집계점포수": "동네_영업기간계산점포수",
    "행정동동종영업점수": "동네_동종점포수",
    "구군동종영업점수": "구군_동종점포수",
    "업종다양성집계점포수": "동네_다양성계산점포수",
}

for region, name in [
    ("행정동", "동네"),
    ("구군", "구군"),
]:
    easy_names.update({
        f"{region}동종프랜차이즈비중보정전":
            f"{name}동종_프랜차이즈비율_원래값",
        f"{region}최근육개월폐업률보정전":
            f"{name}동종_6개월폐업률_원래값",
        f"{region}최근육개월개업률보정전":
            f"{name}동종_6개월개업률_원래값",
        f"{region}최근육개월점포순증보정전":
            f"{name}동종_6개월점포증감_원래값",
        f"{region}폐업점포평균영업개월보정전":
            f"{name}동종_폐업전평균영업기간_원래값",
        f"{region}개업코호트육개월생존율보정전":
            f"{name}동종_신규점포생존율_원래값",

        f"{region}동종프랜차이즈여부관측점포수":
            f"{name}동종_프랜차이즈확인점포수",
        f"{region}동종육개월전영업점수":
            f"{name}동종_6개월전점포수",
        f"{region}동종최근육개월개업점수":
            f"{name}동종_6개월개업수",
        f"{region}동종최근육개월폐업점수":
            f"{name}동종_6개월폐업수",
        f"{region}동종최근일년폐업점수":
            f"{name}동종_1년폐업수",
        f"{region}동종생존율집계개업점수":
            f"{name}동종_생존율계산점포수",
    })

new_columns = [
    easy_names.get(col, col)
    for col in df.columns
]

if len(new_columns) != len(set(new_columns)):
    raise ValueError("변경할 컬럼명이 중복됩니다.")

df.columns = new_columns

display(df.head())
print(df.columns.tolist())

In [ ]:
df["업태구분명"]